# 模块 00：环境准备与图像基础

欢迎来到图像算法实战教学！本模块带你从零开始，理解数字图像在计算机中的表示。

我们将使用：NumPy（图像即数组）、Matplotlib（可视化）、scikit-image（算法库）、OpenCV（工业级CV）、ipywidgets（交互探索）

> 预计学习时间：60-90 分钟

## 学习目标

- 理解图像在 NumPy 中的表示（三维数组 HxWxC）
- 掌握彩色图像的 RGB 通道分解
- 学会 RGB、Grayscale、HSV 之间的转换
- 能够绘制和解读图像直方图
- 掌握图像的基本像素操作（裁剪、翻转、旋转）

## 1. 环境检查与库导入

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import skimage
from skimage import data, color, exposure, io, util, filters
import cv2
import ipywidgets as widgets
from ipywidgets import interact, IntSlider, FloatSlider
from IPython.display import display

print(f"NumPy: {np.__version__}")
print(f"skimage: {skimage.__version__}")
print(f"OpenCV: {cv2.__version__}")
print("All imports OK!")

## 2. 图像即数组——NumPy的核心地位

在计算机中，彩色图像是**三维 NumPy 数组**：
- 维度0 (H,高度)：行数
- 维度1 (W,宽度)：列数
- 维度2 (C,通道)：彩色图=3(R,G,B)，灰度图=1

一张 1920x1080 的彩色照片 → `(1080, 1920, 3)` 的 uint8 数组

In [ ]:
# 加载 skimage 内置示例图片
camera_img = data.camera()       # 灰度图
coffee_img = data.coffee()       # 彩色图
astro_img = data.astronaut()     # 彩色图

print("Camera (灰度):", camera_img.shape, camera_img.dtype, f"range [{camera_img.min()},{camera_img.max()}]")
print("Coffee (彩色):", coffee_img.shape, coffee_img.dtype)
print("Astronaut (彩色):", astro_img.shape, astro_img.dtype)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(camera_img, cmap='gray')
axes[0].set_title(f'camera\n{camera_img.shape}', fontsize=12)
axes[0].axis('off')
axes[1].imshow(coffee_img)
axes[1].set_title(f'coffee\n{coffee_img.shape}', fontsize=12)
axes[1].axis('off')
axes[2].imshow(astro_img)
axes[2].set_title(f'astronaut\n{astro_img.shape}', fontsize=12)
axes[2].axis('off')
plt.tight_layout()
plt.show()

## 3. RGB 通道分解

彩色图像由三个独立的灰度"图层"叠加——红(R)、绿(G)、蓝(B)。每个通道是一张独立的灰度图。

In [ ]:
img = data.astronaut()

# 提取各通道
R, G, B = img[:,:,0], img[:,:,1], img[:,:,2]

# 创建纯色通道图
R_only = np.zeros_like(img); R_only[:,:,0] = R
G_only = np.zeros_like(img); G_only[:,:,1] = G
B_only = np.zeros_like(img); B_only[:,:,2] = B

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].imshow(img)
axes[0].set_title('Original', fontsize=13); axes[0].axis('off')
axes[1].imshow(R_only)
axes[1].set_title('R channel only', fontsize=13); axes[1].axis('off')
axes[2].imshow(G_only)
axes[2].set_title('G channel only', fontsize=13); axes[2].axis('off')
axes[3].imshow(B_only)
axes[3].set_title('B channel only', fontsize=13); axes[3].axis('off')
plt.tight_layout()
plt.show()
print("Key insight: Each channel is an independent grayscale image")

## 4. RGB → 灰度转换

$$Gray = 0.2989 \times R + 0.5870 \times G + 0.1140 \times B$$

人眼对绿色最敏感(59%)，红色次之(30%)，蓝色最不敏感(11%)。

In [ ]:
img = data.astronaut()
img_float = img / 255.0
R, G, B = img_float[:,:,0], img_float[:,:,1], img_float[:,:,2]

# Manual weighted sum
gray_manual = np.clip(0.2989*R + 0.5870*G + 0.1140*B, 0, 1)
# skimage built-in
gray_ski = color.rgb2gray(img)
# Simple average (for comparison)
gray_avg = np.mean(img_float, axis=2)

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
axes[0].imshow(img); axes[0].set_title('Original',fontsize=12); axes[0].axis('off')
axes[1].imshow(gray_manual, cmap='gray')
axes[1].set_title('Weighted (0.299R+0.587G+0.114B)',fontsize=10); axes[1].axis('off')
axes[2].imshow(gray_ski, cmap='gray')
axes[2].set_title('skimage.color.rgb2gray',fontsize=10); axes[2].axis('off')
axes[3].imshow(gray_avg, cmap='gray')
axes[3].set_title('Simple average (darker)',fontsize=10); axes[3].axis('off')
plt.tight_layout()
plt.show()

diff = np.abs(gray_manual - gray_ski).max()
print(f"Manual vs skimage max diff: {diff:.8f}")

## 5. RGB → HSV 颜色空间

HSV 是更符合人直觉的颜色空间：
- **H (Hue色调)**：颜色种类，0°=红，120°=绿，240°=蓝
- **S (Saturation饱和度)**：鲜艳程度，0=灰，1=最艳
- **V (Value明度)**：亮度，0=黑，1=最亮

In [ ]:
img = data.astronaut()
hsv = color.rgb2hsv(img)
H, S, V = hsv[:,:,0], hsv[:,:,1], hsv[:,:,2]

fig, axes = plt.subplots(1, 4, figsize=(18,5))
axes[0].imshow(img); axes[0].set_title('Original RGB',fontsize=13); axes[0].axis('off')
axes[1].imshow(H, cmap='hsv'); axes[1].set_title('H - Hue',fontsize=13); axes[1].axis('off')
axes[2].imshow(S, cmap='gray'); axes[2].set_title('S - Saturation',fontsize=13); axes[2].axis('off')
axes[3].imshow(V, cmap='gray'); axes[3].set_title('V - Value',fontsize=13); axes[3].axis('off')
plt.tight_layout()
plt.show()
print("Observe: white spacesuit = low S (grayish), red helmet = low H value")

## 6. 图像直方图

直方图统计每个像素值出现的频率，反映图像的亮度和对比度分布。

In [ ]:
img = data.camera()

fig, axes = plt.subplots(1, 2, figsize=(14,5))
axes[0].imshow(img, cmap='gray')
axes[0].set_title(f'camera (mean={img.mean():.0f}, std={img.std():.0f})',fontsize=12)
axes[0].axis('off')

axes[1].hist(img.ravel(), bins=256, range=(0,255), color='gray', alpha=0.8)
axes[1].axvline(img.mean(), color='red', linestyle='--', label=f'Mean={img.mean():.0f}')
axes[1].set_title('Grayscale Histogram',fontsize=13)
axes[1].set_xlabel('Pixel value'); axes[1].set_ylabel('Count')
axes[1].legend()
plt.tight_layout()
plt.show()

# RGB histogram
img_c = data.astronaut()
fig, ax = plt.subplots(figsize=(12,5))
for i, c in enumerate(['red','green','blue']):
    ax.hist(img_c[:,:,i].ravel(), bins=256, range=(0,255), color=c, alpha=0.5, label=f'{c.upper()}')
ax.set_title('RGB 3-Channel Histogram',fontsize=13)
ax.set_xlabel('Pixel value'); ax.set_ylabel('Count'); ax.legend()
plt.show()

## 7. 基本像素操作

NumPy 数组操作直接可用于图像处理。

In [ ]:
img = data.astronaut()

# Various operations
crop = img[:200, :200, :]                      # Crop top-left 200x200
flip_h = np.fliplr(img)                        # Horizontal flip
flip_v = np.flipud(img)                        # Vertical flip
rot90 = np.rot90(img)                          # Rotate 90 degrees
bright = np.clip(img.astype(int)+50,0,255).astype(np.uint8)  # Brighten
invert = 255 - img                             # Invert colors

fig, axes = plt.subplots(2, 3, figsize=(15,10))
titles = ['Original','Crop [:200,:200]','Flip H','Flip V','Rotate 90','Brighten +50']
images = [img, crop, flip_h, flip_v, rot90, bright]
for ax, t, im in zip(axes.flatten(), titles, images):
    ax.imshow(im); ax.set_title(t,fontsize=12); ax.axis('off')
plt.tight_layout()
plt.show()

## 🎮 交互式实验

拖动滑块实时调整图像亮度：

In [ ]:
def adjust_brightness(delta=0):
    img = data.camera()
    adjusted = np.clip(img.astype(int)+delta, 0,255).astype(np.uint8)
    fig,(ax1,ax2) = plt.subplots(1,2,figsize=(12,4))
    ax1.imshow(img,cmap='gray'); ax1.set_title(f'Original\nmean={img.mean():.0f}',fontsize=12); ax1.axis('off')
    ax2.imshow(adjusted,cmap='gray'); ax2.set_title(f'Offset {delta:+d}\nmean={adjusted.mean():.0f}',fontsize=12); ax2.axis('off')
    plt.show()

interact(adjust_brightness, delta=IntSlider(min=-100,max=100,step=5,value=0))
print("Slide to adjust brightness offset")

## 📝 应用案例：CT医学影像窗宽窗位调节

CT图像用12-16位深度存储(0-4095+)，人眼无法分辨。放射科医生用**窗宽(Window Width)**和**窗位(Window Level)**调节对比度来突出特定组织。

- **窗位(WL)**：目标组织的平均CT值
- **窗宽(WW)**：显示的CT值范围

In [ ]:
def apply_window(img, level, width):
    low, high = level - width/2, level + width/2
    return np.clip((img - low)/(high - low)*255, 0,255).astype(np.uint8)

img = data.camera()
print(f"Original pixel range: [{img.min()}, {img.max()}]")

params = [(128,256,'Wide: Show all\n(level=128,width=256)'),
          (100,80,'Narrow: High contrast\n(level=100,width=80)'),
          (180,60,'Bright: Highlight bright\n(level=180,width=60)'),
          (60,60,'Dark: Highlight dark\n(level=60,width=60)')]

fig, axes = plt.subplots(1,4,figsize=(18,5))
for ax,(l,w,t) in zip(axes, params):
    ax.imshow(apply_window(img,l,w),cmap='gray')
    ax.set_title(t,fontsize=11); ax.axis('off')
plt.suptitle('CT Window Width/Level Simulation',fontsize=14,y=1.02)
plt.tight_layout()
plt.show()
print("Key insight: Same image, different window settings reveal different details")

## 🏋️ 动手练习

1. **通道交换**：交换 astronaut 图片的 R 和 B 通道，观察颜色变化
2. **自定义灰度**：尝试不同的 RGB 权重（如等权平均），对比与标准公式的差异
3. **直方图分析**：分析 coffee 图片的三通道直方图，判断哪个通道主导了咖啡杯颜色
4. **区域统计**：计算 astronaut 图片中心 200x200 区域的平均 RGB 值

In [ ]:
# TODO: Exercise 1 - Swap R and B channels
# your code here
# TODO: Exercise 4 - Center 200x200 region mean RGB
# center_region = img[..., ...]
print("Complete the exercises above!")

## 📖 本节总结

| 概念 | 要点 |
|------|------|
| 图像=数组 | 彩色(H,W,3) uint8, 灰度(H,W) uint8 |
| RGB | 三个独立灰度图层, 值0-255 |
| 颜色空间 | Gray=加权平均, HSV=人直觉空间 |
| 直方图 | 像素值分布统计 |
| dtype | uint8[0,255] vs float[0,1] |

下一课：模块01 — 滤波与图像增强